In [ ]:
import os
import json
import re
import numpy as np
import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModel
from peft import PeftModel

In [ ]:
MODEL_DIR = "./light_best_model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 1024

print("DEVICE:", DEVICE)

In [ ]:
def extract_grammar_features(text: str):
    text = str(text).strip()
    text_lower = text.lower()

    # words
    words = re.findall(r"\b[\w']+\b", text)
    word_count = max(len(words), 1)

    # sentences
    sentence_candidates = re.split(r"[.!?]+", text)
    sentences = [s.strip() for s in sentence_candidates if s.strip()]
    sentence_count = max(len(sentences), 1)

    sent_word_counts = [len(re.findall(r"\b[\w']+\b", s)) for s in sentences]
    if len(sent_word_counts) == 0:
        sent_word_counts = [1]

    avg_sentence_len = word_count / sentence_count
    short_ratio = sum(1 for c in sent_word_counts if c < 8) / sentence_count
    long_ratio  = sum(1 for c in sent_word_counts if c > 25) / sentence_count

    # punctuation / simple complexity proxies
    comma_count = text.count(",")
    semicolon_count = text.count(";")
    colon_count = text.count(":")

    # discourse / grammar proxies
    conj_count = len(re.findall(
        r"\b(and|but|or|because|although|though|while|whereas|if|when|since|unless|however|therefore)\b",
        text_lower
    ))

    pron_count = len(re.findall(
        r"\b(i|you|we|they|he|she|it|this|that|these|those)\b",
        text_lower
    ))

    modal_count = len(re.findall(
        r"\b(can|could|may|might|must|should|would|will)\b",
        text_lower
    ))

    article_count = len(re.findall(r"\b(a|an|the)\b", text_lower))

    be_verb_count = len(re.findall(
        r"\b(am|is|are|was|were|be|been|being)\b",
        text_lower
    ))

    feats = np.array([
        word_count,        # 1
        sentence_count,    # 2
        avg_sentence_len,  # 3
        short_ratio,       # 4
        long_ratio,        # 5
        comma_count,       # 6
        semicolon_count,   # 7
        colon_count,       # 8
        conj_count,        # 9
        pron_count,        # 10
        modal_count,       # 11
    ], dtype=np.float32)

    return feats

In [ ]:
class IELTSModel(nn.Module):
    def __init__(self, backbone, feat_dim=5, dropout=0.1):
        super().__init__()

        self.backbone = backbone
        hidden = backbone.config.hidden_size

        self.dropout = nn.Dropout(dropout)

        self.head_tr = nn.Linear(hidden, 1)
        self.head_cc = nn.Linear(hidden, 1)
        self.head_lr = nn.Linear(hidden, 1)

        self.gra_mlp = nn.Sequential(
            nn.Linear(hidden + feat_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

        self.act = nn.Sigmoid()

    def last_token_pool(self, last_hidden_state, attention_mask):
        # lấy hidden state của token cuối cùng còn hiệu lực
        idx = attention_mask.sum(dim=1) - 1
        idx = idx.clamp(min=0)

        batch_idx = torch.arange(last_hidden_state.size(0), device=last_hidden_state.device)
        pooled = last_hidden_state[batch_idx, idx]
        return pooled

    def forward(self, input_ids, attention_mask, gra_features):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        hidden = outputs.last_hidden_state
        pooled = self.last_token_pool(hidden, attention_mask)
        pooled = self.dropout(pooled)

        tr = self.act(self.head_tr(pooled))
        cc = self.act(self.head_cc(pooled))
        lr = self.act(self.head_lr(pooled))

        gra_input = torch.cat([pooled, gra_features], dim=1)
        gra = self.act(self.gra_mlp(gra_input))

        preds = torch.cat([tr, cc, lr, gra], dim=1)  # shape [B, 4]
        return preds

In [ ]:
meta_path = os.path.join(MODEL_DIR, "export_meta.json")

with open(meta_path, "r", encoding="utf-8") as f:
    export_meta = json.load(f)

export_meta

In [ ]:
# linh hoạt vì có thể notebook export key hơi khác tên
base_model_name = (
    export_meta.get("base_model_name")
    or export_meta.get("model_name")
    or export_meta.get("backbone_name")
    or "Qwen/Qwen2.5-3B-Instruct"
)

max_length = export_meta.get("max_length", MAX_LENGTH)

gra_feat_mean = np.array(
    export_meta.get("gra_feat_mean", [0, 0, 0, 0, 0]),
    dtype=np.float32
)

gra_feat_std = np.array(
    export_meta.get("gra_feat_std", [1, 1, 1, 1, 1]),
    dtype=np.float32
)

# tránh chia cho 0
gra_feat_std = np.where(gra_feat_std == 0, 1.0, gra_feat_std)

print("base_model_name =", base_model_name)
print("max_length      =", max_length)
print("gra_feat_mean   =", gra_feat_mean)
print("gra_feat_std    =", gra_feat_std)

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
base_model = AutoModel.from_pretrained(
    base_model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
)

print("Loading LoRA adapter...")
backbone = PeftModel.from_pretrained(base_model, MODEL_DIR)

heads_path = os.path.join(MODEL_DIR, "custom_heads.pt")
print("Loading heads from:", heads_path)
head_state = torch.load(heads_path, map_location="cpu")

hidden_size = backbone.config.hidden_size
saved_in_dim = head_state["gra_mlp"]["0.weight"].shape[1]
feat_dim = saved_in_dim - hidden_size

print("hidden_size =", hidden_size)
print("saved gra_mlp input dim =", saved_in_dim)
print("=> inferred feat_dim =", feat_dim)

model = IELTSModel(backbone, feat_dim=feat_dim, dropout=0.1)

model.head_tr.load_state_dict(head_state["head_tr"])
model.head_cc.load_state_dict(head_state["head_cc"])
model.head_lr.load_state_dict(head_state["head_lr"])
model.gra_mlp.load_state_dict(head_state["gra_mlp"])

model = model.to(DEVICE)

if DEVICE == "cuda":
    model = model.half()

model.eval()

print("Model loaded successfully.")
print("Model dtype:", next(model.parameters()).dtype)

In [ ]:
def predict(prompt: str, essay: str):
    prompt = str(prompt).strip()
    essay = str(essay).strip()

    text = f"[PROMPT]\n{prompt}\n\n[ESSAY]\n{essay}"

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        padding=False,
        return_tensors="pt"
    )

    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    gra_feats = extract_grammar_features(essay)
    assert gra_feats.shape[0] == len(gra_feat_mean), f"Expected {len(gra_feat_mean)} grammar features, got {gra_feats.shape[0]}"
    gra_feats = (gra_feats - gra_feat_mean) / gra_feat_std

    model_dtype = next(model.parameters()).dtype
    gra_feats = torch.tensor(gra_feats, dtype=model_dtype).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        preds = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            gra_features=gra_feats
        )

    preds = preds[0].detach().float().cpu().numpy()
    scores = preds * 9.0
    scores = np.clip(scores, 0.0, 9.0)
    scores = np.round(scores * 2) / 2.0

    tr, cc, lr, gra = scores.tolist()
    overall = round((tr + cc + lr + gra) / 4 * 2) / 2.0

    return {
        "TR": tr,
        "CC": cc,
        "LR": lr,
        "GRA": gra,
        "Overall": overall
    }

In [ ]:
prompt = "Some people think that children should begin formal education at a very early age. To what extent do you agree or disagree?"

essay = """
In my opinion, children should not begin formal education too early. Although learning basic skills at a young age can be helpful,
forcing children into an academic environment too soon may create stress and reduce their interest in learning.

First, young children learn effectively through play and social interaction. Activities such as drawing, storytelling and group games
help them develop creativity, communication and emotional intelligence. If formal schooling starts too early, these natural forms of
learning may be replaced by pressure to perform academically.

Second, early formal education can negatively affect mental well-being. Some children may struggle to adapt to strict schedules and
classroom discipline, which can lead to anxiety or low confidence. A more balanced approach would allow children to mature emotionally
before facing serious academic demands.

In conclusion, while early exposure to learning can be beneficial, I believe formal education should not start too early because
children need time to develop socially and emotionally first.
"""

result = predict(prompt, essay)
result

In [ ]:
while True:
    prompt = input("Enter prompt (or 'quit'): ").strip()
    if prompt.lower() == "quit":
        break

    print("Paste essay below. End with a line containing only: END")
    lines = []
    while True:
        line = input()
        if line.strip() == "END":
            break
        lines.append(line)

    essay = "\n".join(lines)

    result = predict(prompt, essay)
    print(result)
    print("-" * 60)